# 2 · PINN with Bayesian hyperparameter optimisation (no FiLM)

Standard Fourier-feature MLP PINN; loss weights, regularisation penalties and network size are tuned with Optuna.

Hyperparameters are selected with Optuna (TPE sampler) by maximising the **Kling–Gupta efficiency on the validation
partition**. The test partition is not used by the objective; with `RETRAIN = True` the best configuration is retrained and
scored once on the test partition.

**Before running:** place the input files in `../data/` as described in [`data/README.md`](../data/README.md), or generate a
synthetic dataset with `python examples/make_synthetic_data.py --out data` from the repository root.

A full study is computationally expensive (each trial trains a PINN for several thousand iterations); a GPU is strongly
recommended.

In [ ]:
import os, sys

# Make the repository root importable and point the module at the input data.
REPO_ROOT = os.path.abspath("..")
sys.path.insert(0, REPO_ROOT)
DATA_DIR = os.path.join(REPO_ROOT, "data")      # see data/README.md for the required files
os.environ["DDE_BACKEND"] = "pytorch"           # must be set before deepxde is imported

import pinn_unified as pu

## Configuration

`SPLIT_TYPE` is one of `'random'`, `'temporal'` or `'temporal_lowdata'`.

In [ ]:
SPLIT_TYPE = 'temporal'
USE_FILM = False
N_TRIALS = 10
STUDY_NAME = 'github_optuna_no_film'
STORAGE = f'sqlite:///{STUDY_NAME}.db'   # persistent Optuna study (resumable)
OUTPUT_DIR = f'results_{STUDY_NAME}_{SPLIT_TYPE}'
RETRAIN = False                          # True: retrain best trial and score once on the test partition

## 1. Load and scale the data

In [ ]:
stage_s, gwt_s, well_s, stats_df = pu.load_and_preprocess_data(data_dir=DATA_DIR)

## 2. Run the Bayesian optimisation

In [ ]:
study, retrain_results = pu.optimize_pinn(
    study_name=STUDY_NAME,
    n_trials=N_TRIALS,
    storage=STORAGE,
    split_type=SPLIT_TYPE,
    use_film=USE_FILM,
    fixed_split_seed=42,
    output_dir=OUTPUT_DIR,
    retrain=RETRAIN,
)
print("Best validation KGE:", study.best_value)
print("Best hyperparameters:", study.best_params)